# CROSS step 1 on Kaggle: plane-matched donors and more matched controls

**Inputs (Add Input):** (1) `masoudnickparvar/brain-tumor-mri-dataset`; (2) your private dataset made from
`cross_bundle.zip` (version with `results/audit/planes.csv`). **Settings:** GPU T4, Internet **on**.
Run with *Save Version → Save & Run All (Commit)*. Outputs: `/kaggle/working/cross_kaggle/` → `results/cross_kaggle/`.


In [ ]:
import glob, os, shutil, subprocess
def find(rel):
    # fixed-depth search (a recursive glob over the mounted inputs takes minutes)
    for depth in range(0, 5):
        hits = sorted(glob.glob("/kaggle/input/" + "*/" * depth + rel))
        if hits:
            return hits[0]
    raise FileNotFoundError(rel)
bundle = find("results/audit/grouped_folds.csv")[: -len("results/audit/grouped_folds.csv")].rstrip("/")
data = find("Training/glioma")[: -len("Training/glioma")].rstrip("/")   # the folder that holds Training/ and Testing/
assert os.path.isdir(f"{data}/Testing"), data
P = "/kaggle/working/proj"
if not os.path.exists(P):
    shutil.copytree(bundle, P)
os.makedirs(f"{P}/data", exist_ok=True)
if not os.path.exists(f"{P}/data/brain-tumor-mri-dataset"):
    os.symlink(data, f"{P}/data/brain-tumor-mri-dataset")
print("bundle:", bundle, "| data:", data)
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
import sys
env = dict(os.environ, CROSS_OUT="/kaggle/working/cross_kaggle", PYTHONUNBUFFERED="1")
def go(arms, seeds):
    cmd = [sys.executable, "code/cross_pilot.py", "--arms", arms, "--folds", "0,1,2,3,4", "--seeds", seeds]
    p = subprocess.Popen(cmd, cwd=P, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    assert p.wait() == 0
# QUICK_TEST: one 1-epoch run to check the setup (writes to a separate folder)
QUICK_TEST = False
if QUICK_TEST:
    env["CROSS_OUT"] = "/kaggle/working/cross_quick"
    subprocess.run([sys.executable, "code/cross_pilot.py", "--arms", "cross_pm_r75_c67", "--folds", "0", "--seeds", "1", "--epochs", "1"], cwd=P, env=env, check=True)
else:
    # 30 runs: plane-matched donors (cross_pm), and plane-matched with more controls (25% transplants, 50% controls)
    go("cross_pm,cross_pm_r75_c67", "42,43,44")
    go("cross_p", "42")                      # 5 runs: the pilot arm at seed 42 on the same hardware


In [ ]:
shutil.make_archive("/kaggle/working/cross_pm_results", "zip", "/kaggle/working/cross_kaggle")
print(open("/kaggle/working/cross_kaggle/results.csv").read())
